In [1]:
import os
from dotenv import load_dotenv, find_dotenv 

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "Agents"

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings 
llm = ChatOpenAI() 
embedding = OpenAIEmbeddings()

#### **Custom Tools**

In [3]:
from langchain.agents import tool

@tool 
def get_word_length(word : str) -> int:
    """Returns the length of the word"""
    return len(word)

In [4]:
get_word_length.invoke("Hello World, Welcome to programming")

35

In [5]:
@tool 
def multiply(a: int, b : int) -> int:
    """Multiply two numbers"""
    return a * b

In [6]:
@tool 
def summation(a: int, b : int) -> int: 
    """Adding 2 numbers """
    return a + b 

In [7]:
print(multiply.name)
print(multiply.description)
print(multiply.args)


multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [8]:
multiply.invoke({"a": 10, "b" : 20})

200

In [9]:
summation.invoke({"a" : 1000, "b" : 1000})

2000

#### **Agents**

In [10]:
from langchain.agents import AgentType 
from langchain.agents import load_agent 
from langchain.agents import initialize_agent 
from langchain.agents import load_tools 

In [11]:
tool = load_tools(['wikipedia'], llm = llm)

In [12]:
agent = initialize_agent(tool, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose = True)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_12396\220137838.py:1: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use :meth:`~Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc.` instead.
  agent = initialize_agent(tool, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose = True)


In [13]:
agent.run("What is current GDP of India")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_12396\2435368785.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent.run("What is current GDP of India")




> Entering new AgentExecutor chain...
you need to find the most recent information on India's GDP
Action: wikipedia
Action Input: GDP of India
Observation: Page: Economy of India
Summary: The economy of India is a developing mixed economy with a notable public sector in strategic sectors. It is the world's fifth-largest economy by nominal GDP and the third-largest by purchasing power parity (PPP); on a per capita income basis, India ranked 141th by GDP (nominal) and 125th by GDP (PPP). From independence in 1947 until 1991, successive governments followed the Soviet model and promoted protectionist economic policies, with extensive Sovietization, state intervention, demand-side economics, natural resources, bureaucrat driven enterprises and economic regulation. This is characterised as dirigism, in the form of the Licence Raj. The end of the Cold War and an acute balance of payments crisis in 1991 led to the adoption of a broad economic liberalisation in India and indicative planning.

"The current GDP of India is the world's fifth-largest economy by nominal GDP and the third-largest by purchasing power parity (PPP)"

In [14]:
agent.run("What is LLama")



> Entering new AgentExecutor chain...
I should look up the definition of llama to provide an accurate answer
Action: wikipedia
Action Input: Llama

d:\Code\Langchain\venv\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file d:\Code\Langchain\venv\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')



Observation: Page: Llama
Summary: The llama (; Spanish pronunciation: [ˈʎama] or [ˈʝama]) (Lama glama) is a domesticated South American camelid, widely used as a meat and pack animal by Andean cultures since the pre-Columbian era.
Llamas are social animals and live with others as a herd. Their wool is soft and contains only a small amount of lanolin. Llamas can learn simple tasks after a few repetitions. When using a pack, they can carry about 25 to 30% of their body weight for 8 to 13 km (5–8 miles). The name llama (in the past also spelled "lama" or "glama") was adopted by European settlers from native Peruvians.
The ancestors of llamas are thought to have originated on the Great Plains of North America about 40 million years ago and subsequently migrated to South America about three million years ago during the Great American Interchange. By the end of the last ice age (10,000–12,000 years ago), camelids were extinct in North America. As of 2007, there were over seven million llama

'A llama is a domesticated South American camelid widely used as a meat and pack animal by Andean cultures since the pre-Columbian era. It is also the name of a family of autoregressive large language models released by Meta AI starting in February 2023.'

In [15]:
from langchain_community.tools.tavily_search import TavilySearchResults 
search = TavilySearchResults()

In [16]:
search.invoke("What is LLama 3.2")

"HTTPError('401 Client Error: Unauthorized for url: https://api.tavily.com/search')"

In [20]:
tools = [search]

In [21]:
from langchain import hub

In [23]:
prompt = hub.pull("hwchase17/openai-functions-agent")

In [24]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [25]:
from langchain.agents import create_tool_calling_agent
agent = create_tool_calling_agent(llm, tools, prompt)

In [26]:
from langchain.agents import AgentExecutor 
agent_executor = AgentExecutor(agent = agent, tools = tools, verbose=True)

In [27]:
agent_executor.invoke({"input": "Hello, how are you ?"})



> Entering new AgentExecutor chain...
Hello! I'm here and ready to assist you. How can I help you today?

> Finished chain.


{'input': 'Hello, how are you ?',
 'output': "Hello! I'm here and ready to assist you. How can I help you today?"}